# Phase 4：訂單整合分析（時間序列版 v3 - 含客戶分類與路徑分析）

使用「成交前最後一次商機評估」進行時間序列對應

## 核心改進

**v1（舊版）問題：**
- ❌ 只看 company_id 對應
- ❌ 不管訂單何時成交，只要有 A 階段就算「A 階段成交」
- ❌ 無法知道真正成交時的階段

**v2（時間序列版）改進：**
- ✅ 找「成交前最後一次商機評估」
- ✅ 計算「評估 → 成交」的天數
- ✅ 分析各階段真實成交週期

**v3（本版）新增：**
- ✅ **客戶分類**：新客戶 vs 老客戶 / 大客戶 vs 小客戶
- ✅ **商機路徑分析**：追蹤每個法人的完整階段流轉路徑（排除同階段重複）
- ✅ **熱門路徑統計**：找出最常見的成交路徑（例如：E → D → C2 → C1 → B → A）
- ✅ **反向流程分析**：檢測 A→C1、B→C2 等反向流程，判斷是否為老客戶重購或記錄錯誤
- ✅ **單階段成交分析**：分析只有單一階段記錄的成交案例
- ✅ **成交路徑分析**：追蹤每筆訂單從第一次接觸到成交的完整歷程（例如：E → D → C1 【A成交】）
- ✅ **樹狀路徑分析**：建立樹狀結構展示所有可能路徑，標記成交路徑與再購路徑
- ✅ **圖表改進**：
  - 圖1：堆疊橫條圖顯示百分比
  - 圖3：散佈圖按客戶類型著色
  - 圖4：箱型圖比較新 vs 老客戶
  - 圖7（新增）：Sankey 圖展示商機流轉路徑（排除同階段重複，只顯示真實階段變化）
  - 圖8（新增）：Sunburst 圖展示樹狀成交路徑（紅色=高再購，藍色=高新客）

---

## 輸出檔案

- 📊 **15 個 CSV**：
  - C1_stage_conversion_timeseries.csv - 各階段成交統計 + 平均週期
  - C2_cycle_distribution.csv - 成交週期分布
  - matched_orders.csv - 完整匹配結果（含客戶分類）
  - stage_transitions.csv - 商機階段轉換矩陣（排除同階段重複）
  - company_stage_paths.csv - 每家公司的完整階段路徑
  - hot_stage_paths.csv - 熱門路徑統計（含出現次數與百分比）
  - reverse_transitions.csv - 反向流程明細（階段降級）
  - single_vs_multi_stage.csv - 單階段 vs 多階段成交統計
  - order_conversion_paths.csv - 每筆訂單的完整成交路徑（含起點、終點、路徑長度）
  - hot_conversion_paths.csv - 熱門成交路徑排行榜
  - **tree_path_analysis.csv - 完整樹狀路徑分析（含部分路徑與完整路徑）**
  - **conversion_leaf_paths.csv - 所有成交路徑（葉子節點）**
  - **repurchase_dominated_paths.csv - 高再購路徑（再購率≥70%）**
  - **new_customer_dominated_paths.csv - 高新客路徑（新客率≥70%）**
- 📈 **圖表**：
  - Phase4_Complete_Visualization_v3.png - 6 張圖（含客戶分類分析）
  - Stage_Flow_Sankey.html - 商機流轉路徑圖（互動式，排除同階段重複）
  - **Tree_Path_Sunburst.html - 樹狀成交路徑圖（互動式旭日圖，紅/藍/黃著色）**

---

## 關鍵改進說明

### 🔥 排除同階段重複
過去的分析包含了大量「C1 → C1」、「B → B」等同階段重複記錄（佔總記錄的 ~30-40%），這些並非真正的階段晉升，而是業務人員在同一階段的多次更新。

**v3 版本改進：**
- 只保留真實的階段變化（例如：E → D, D → C2, C2 → C1）
- 排除同階段重複（C1 → C1, B → B, A → A）
- Sankey 圖更清楚展示客戶的真實晉升路徑

### 📊 完整路徑追蹤
為每個成交法人建立從第一次出現到最後的完整階段路徑，例如：
- E → D → C2 → C1 → B → A（完整晉升）
- C2 → C1 → B（部分晉升）
- A（直接成交）

統計最熱門的路徑組合，了解客戶最常見的成交旅程。

### 🔍 反向流程分析
檢測階段降級現象（A→C1, B→C2 等），並根據時間間隔判斷原因：
- **<7天**：可能是記錄錯誤或階段調整
- **7-90天**：可能是新專案評估或業務追蹤
- **>90天**：很可能是老客戶重新採購

交叉比對訂單資料，驗證是否為多筆訂單客戶（老客戶重購）。

### 🎯 單階段成交分析
分析只有單一階段記錄就成交的案例（C1、A、B、D 單階段成交），交叉比對：
- 新客戶 vs 老客戶比例
- 大客戶 vs 小客戶分布
- 與多階段成交的差異

了解哪些客戶適合快速成交，哪些需要長期培養。

### 🎯 成交路徑分析
**區分「轉換路徑」vs「成交路徑」：**

- **轉換路徑**（stage_transitions.csv）：
  - 展示階段之間的流轉關係
  - 例如：C1 → B（408次）、B → C1（381次）
  - 用途：了解客戶在各階段間如何移動

- **成交路徑**（order_conversion_paths.csv）：
  - 展示每筆訂單從開始到成交的完整歷程
  - 例如：E → D → C2 → C1 → B 【A成交】
  - 用途：了解成交訂單的完整培養過程

**分析內容：**
1. 熱門成交路徑排行榜
2. 路徑長度分布（1階段直接成交 vs 5+階段完整培養）
3. 各成交階段的平均路徑長度
4. 起始階段 → 成交階段組合分析
5. 直接成交 vs 完整培養的客戶特徵差異

### 🌳 樹狀路徑分析（新增）
建立完整的樹狀結構，展示所有可能的成交路徑，並自動標記：

**✅ 成交路徑標記：**
- 所有最終成交的路徑（葉子節點）
- 顯示每條路徑的訂單數量
- 標示成交階段

**🔄 再購路徑標記：**
- **高再購路徑**（再購率≥70%）：🔴 紅色標記
  - 主要是老客戶重複購買的路徑
  - 適合設計再購促銷方案
  
- **高新客路徑**（新客率≥70%）：🔵 藍色標記
  - 主要是新客戶首次購買的路徑
  - 適合優化新客獲取流程

- **混合路徑**（30%<再購率<70%）：🟡 黃色標記
  - 新客與老客混合的路徑

**分析內容：**
1. 所有成交路徑總覽（含部分路徑與完整路徑）
2. 前30大成交路徑（按訂單數排序）
3. 高再購路徑TOP 20（再購率≥70%）
4. 高新客路徑TOP 20（新客率≥70%）
5. 路徑長度分析（各長度的再購率與平均金額）
6. 互動式 Sunburst 圖（旭日圖）：
   - 從中心向外展開，展示路徑階層
   - 點擊可深入查看子路徑
   - 懸停顯示詳細資訊（訂單數、新客/再購比例）

---

**執行方式：** Kernel → Restart & Run All

## S0：匯入套件 & 全域設定

In [ ]:
import os
import pathlib
import pyodbc
import configparser
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# CJK 字型
try:
    fm.fontManager.addfont(r"C:\Windows\Fonts\msjh.ttc")
    plt.rcParams["font.family"] = "Microsoft JhengHei"
except Exception:
    pass

# 路徑設定
BASE = pathlib.Path.cwd()
ROOT = BASE.parent if 'Phase4' in str(BASE) else BASE
OUT = BASE / "results_timeseries"
OUT.mkdir(parents=True, exist_ok=True)

print("Phase 4：訂單整合分析（時間序列版）")
print(f"輸出目錄：{OUT}")

## S1：載入基礎資料

In [ ]:
print("載入 Phase 1-3 資料...")

# Phase M：聚類
clusters_df = pd.read_csv(
    ROOT / "PhaseM_聚類分析/results/company_clusters.csv",
    dtype={"company_id": str, "cluster": int}
)
clusters_df = clusters_df[["company_id", "cluster"]].copy()
print(f"   PhaseM 聚類：{len(clusters_df):,} 家")

# Phase 3：痛點明細
pain_records = pd.read_csv(
    ROOT / "Phase3_痛需熱圖/results/pain_records.csv",
    dtype={"company_id": str}
)
print(f"   Phase3 痛點：{len(pain_records):,} 筆 / {pain_records['company_id'].nunique():,} 家")

print("\n資料載入完成")

## S2：載入訂單資料（SQL Server）

In [ ]:
print("載入訂單資料...")

SQL_INI = r"D:\yujui\SqlServer18.txt"
cfg = configparser.ConfigParser()
cfg.read(SQL_INI, encoding="utf-8")
sec = cfg["mssql"]

CONN_STR = (
    f"DRIVER={{SQL Server}};SERVER={sec['server']};"
    f"DATABASE=DSC_CRM_SP;UID={sec['uid']};PWD={sec['pwd']}"
)

conn = pyodbc.connect(CONN_STR, autocommit=True)

order_sql = """
SELECT
    HC015 as company_id,
    MODI_DATE,
    HC018 as order_amount,
    HC043 as product_code
FROM DSC_CRM_SP.dbo.CRMHC
WHERE LEN(HC015) = 10
AND MODI_DATE >= '20240101'
AND HC018 IS NOT NULL
ORDER BY MODI_DATE DESC
"""

orders_df = pd.read_sql(order_sql, conn)
conn.close()

orders_df["order_amount"] = pd.to_numeric(orders_df["order_amount"], errors="coerce")
orders_df["order_date"] = pd.to_datetime(orders_df["MODI_DATE"], format="%Y%m%d", errors="coerce")

print(f"   訂單筆數：{len(orders_df):,}")
print(f"   公司數：{orders_df['company_id'].nunique():,}")
print(f"   日期範圍：{orders_df['order_date'].min()} ~ {orders_df['order_date'].max()}")
print(f"   總金額：{orders_df['order_amount'].sum():,.0f} 元")

print("\n訂單資料載入完成")
orders_df.head()

## S3：載入商機階段（帶日期）

In [ ]:
print("載入商機階段資料（帶日期）...")

stage_path = r"d:\yujui\痛點需求地圖\lead_stage_with_date.csv"
stage_df = pd.read_csv(stage_path, encoding='utf-8-sig')
stage_df['log_date'] = pd.to_datetime(stage_df['log_date'])

print(f"   商機日誌：{len(stage_df):,} 筆")
print(f"   涵蓋公司：{stage_df['company_id'].nunique():,} 家")
print(f"   日期範圍：{stage_df['log_date'].min()} ~ {stage_df['log_date'].max()}")

print("\n各階段日誌數量：")
print(stage_df['stage'].value_counts().sort_index())

print("\n商機階段資料載入完成")
stage_df.head()

---

# 核心邏輯：時間序列對應

為每筆訂單找「成交前最後一次商機評估」

In [ ]:
print("="*70)
print("時間序列對應：訂單 × 商機日誌")
print("="*70)

# 1. 合併訂單與商機日誌（笛卡爾積）
merged = orders_df.merge(
    stage_df[['company_id', 'log_date', 'stage']],
    on='company_id',
    how='left'
)

print(f"\n步驟 1：初步合併：{len(merged):,} 筆（訂單 × 商機日誌）")

# 2. 篩選：只保留「日誌日期 <= 訂單日期」
merged = merged[merged['log_date'] <= merged['order_date']].copy()

print(f"步驟 2：篩選「成交前」日誌：{len(merged):,} 筆")

# 3. 計算時間差（天數）
merged['days_before_order'] = (merged['order_date'] - merged['log_date']).dt.days

# 4. 為每筆訂單找「最接近成交日期」的商機評估
matched_orders = merged.sort_values('days_before_order').groupby(
    ['company_id', 'MODI_DATE', 'order_amount', 'order_date', 'product_code']
).first().reset_index()

print(f"\n步驟 3：最終匹配：{len(matched_orders):,} 筆訂單")
print(f"匹配率：{len(matched_orders) / len(orders_df) * 100:.1f}%")

unmatched = len(orders_df) - len(matched_orders)
print(f"\n無法匹配：{unmatched:,} 筆（{unmatched/len(orders_df)*100:.1f}%）")
print("  原因：成交前無商機日誌記錄")
print("  可能：老客戶重購、快速成交、業務未記錄")

print("\n="*70)

## 匹配結果預覽

In [ ]:
print("匹配結果範例（前 10 筆）：")
display(matched_orders[[
    'company_id', 'order_date', 'order_amount', 
    'stage', 'log_date', 'days_before_order'
]].head(10))

# 儲存完整匹配結果
matched_orders.to_csv(
    OUT / "matched_orders.csv",
    index=False,
    encoding="utf-8-sig"
)
print(f"\n已儲存：matched_orders.csv")

---

# 分析 C1：各階段成交統計（時間序列版）

In [ ]:
print("="*70)
print("分析 C1：各階段成交統計 + 平均週期")
print("="*70)

stage_stats = matched_orders.groupby('stage').agg(
    order_count=('company_id', 'count'),
    total_revenue=('order_amount', 'sum'),
    avg_revenue=('order_amount', 'mean'),
    avg_days_to_close=('days_before_order', 'mean'),
    median_days_to_close=('days_before_order', 'median'),
    min_days=('days_before_order', 'min'),
    max_days=('days_before_order', 'max'),
).reset_index()

STAGE_ORDER = {"A": 1, "B": 2, "C1": 3, "C2": 4, "D": 5, "E": 6}
stage_stats["stage_rank"] = stage_stats["stage"].map(STAGE_ORDER)
stage_stats = stage_stats.sort_values("stage_rank")

print("\n各階段成交統計：")
display(stage_stats[[
    'stage', 'order_count', 'total_revenue', 'avg_revenue',
    'avg_days_to_close', 'median_days_to_close'
]])

stage_stats.to_csv(
    OUT / "C1_stage_conversion_timeseries.csv",
    index=False,
    encoding="utf-8-sig"
)
print(f"\n已儲存：C1_stage_conversion_timeseries.csv")

## 關鍵洞察 ⭐

In [ ]:
print("="*70)
print("關鍵洞察")
print("="*70)

fastest_stage = stage_stats.loc[stage_stats['avg_days_to_close'].idxmin()]
slowest_stage = stage_stats.loc[stage_stats['avg_days_to_close'].idxmax()]
most_orders = stage_stats.loc[stage_stats['order_count'].idxmax()]

print(f"\n1. 最快成交階段：{fastest_stage['stage']}")
print(f"   平均 {fastest_stage['avg_days_to_close']:.0f} 天 / {fastest_stage['order_count']:.0f} 筆訂單")
print(f"   建議：加速推進到 {fastest_stage['stage']} 階段")

print(f"\n2. 最慢成交階段：{slowest_stage['stage']}")
print(f"   平均 {slowest_stage['avg_days_to_close']:.0f} 天 / {slowest_stage['order_count']:.0f} 筆訂單")
print(f"   建議：{slowest_stage['stage']} 階段快速推進，避免停滯")

print(f"\n3. 訂單數最多階段：{most_orders['stage']}")
print(f"   {most_orders['order_count']:.0f} 筆訂單 / 平均 {most_orders['avg_days_to_close']:.0f} 天")

overall_avg = matched_orders['days_before_order'].mean()
overall_median = matched_orders['days_before_order'].median()

print(f"\n4. 整體成交週期：")
print(f"   平均：{overall_avg:.0f} 天")
print(f"   中位數：{overall_median:.0f} 天")
print(f"   差異：{overall_avg - overall_median:.0f} 天 → 有超長週期訂單拉高平均")

print("\n="*70)

---

# 分析 C2：成交週期分布

In [ ]:
print("="*70)
print("分析 C2：成交週期分布")
print("="*70)

# 按天數區間分組
bins = [0, 7, 30, 90, 180, 365, 9999]
labels = ['<7天', '7-30天', '30-90天', '90-180天', '180-365天', '>365天']

matched_orders['cycle_group'] = pd.cut(
    matched_orders['days_before_order'],
    bins=bins,
    labels=labels
)

cycle_stats = matched_orders.groupby(['stage', 'cycle_group']).size().unstack(fill_value=0)

print("\n各階段成交週期分布（訂單數）：")
display(cycle_stats)

# 計算百分比
cycle_pct = cycle_stats.div(cycle_stats.sum(axis=1), axis=0) * 100

print("\n各階段成交週期分布（百分比）：")
display(cycle_pct.round(1))

cycle_stats.to_csv(
    OUT / "C2_cycle_distribution.csv",
    encoding="utf-8-sig"
)
print(f"\n已儲存：C2_cycle_distribution.csv")

---

# 視覺化：成交週期分析

---

# 視覺化：完整分析圖表

## 📊 6 張圖表（含客戶分類分析）

下方將生成 **3×2 網格圖表**，包含：

1. **成交週期分布（堆疊橫條圖 + 百分比）** - 各階段在不同時間區間的訂單分布，標註百分比
2. **時間趨勢分析** - 按月份追蹤訂單數與平均成交週期的變化
3. **金額 vs 週期散佈圖（按客戶類型著色）** - 新/老客戶 × 大/小客戶 4 種類型
4. **各階段成交週期箱型圖（新 vs 老客戶）** - 比較新客戶與老客戶的成交週期差異
5. **成交週期累積分布曲線（CDF）** - 各階段在特定天數內的成交比例
6. **各階段訂單數與平均週期** - 雙軸圖綜合呈現訂單量與週期關係

---

**圖表輸出：** `results_timeseries/Phase4_Complete_Visualization_v3.png` (18×14 吋, 200 DPI)

In [ ]:
# ========== 步驟 1：客戶分類 ==========
print("="*70)
print("客戶分類分析")
print("="*70)

# 為每個 company_id 計算訂單序號（按日期排序）
matched_orders_sorted = matched_orders.sort_values(['company_id', 'order_date'])
matched_orders_sorted['order_sequence'] = matched_orders_sorted.groupby('company_id').cumcount() + 1

# 判斷是否為新客戶（首次下單）
matched_orders_sorted['customer_type'] = matched_orders_sorted['order_sequence'].apply(
    lambda x: '新客戶' if x == 1 else '老客戶'
)

# 判斷大小客戶（100萬為門檻）
matched_orders_sorted['customer_size'] = matched_orders_sorted['order_amount'].apply(
    lambda x: '大客戶' if x >= 1000000 else '小客戶'
)

# 綜合分類
matched_orders_sorted['customer_segment'] = (
    matched_orders_sorted['customer_type'] + ' - ' + 
    matched_orders_sorted['customer_size']
)

print("\n客戶類型分布：")
print(matched_orders_sorted['customer_type'].value_counts())

print("\n客戶規模分布：")
print(matched_orders_sorted['customer_size'].value_counts())

print("\n客戶細分：")
print(matched_orders_sorted['customer_segment'].value_counts())

# 更新 matched_orders 以便後續使用
matched_orders = matched_orders_sorted.copy()

print("\n客戶分類完成！")
print("="*70)

In [ ]:
# 過濾極端值（避免圖表失真）
plot_data = matched_orders[matched_orders['days_before_order'] <= 365].copy()

# 建立 3×2 圖表
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
ax1, ax2, ax3, ax4, ax5, ax6 = axes.flat

# =========== 圖 1：堆疊橫條圖 - 各階段成交週期分布（含百分比標註） ===========
cycle_pct_sorted = cycle_pct.reindex(stage_stats['stage'].values, fill_value=0)

# 繪製堆疊橫條圖
bars = cycle_pct_sorted.plot(
    kind='barh', 
    stacked=True, 
    ax=ax1,
    colormap='RdYlGn_r',
    width=0.7
)

# 在各色塊標註百分比
for i, stage in enumerate(cycle_pct_sorted.index):
    cumsum = 0
    for col in cycle_pct_sorted.columns:
        val = cycle_pct_sorted.loc[stage, col]
        if val > 3:  # 只標註大於 3% 的區塊
            ax1.text(cumsum + val/2, i, f'{val:.1f}%', 
                    ha='center', va='center', fontsize=8, 
                    color='white', fontweight='bold')
        cumsum += val

ax1.set_xlabel('百分比 (%)', fontsize=11)
ax1.set_ylabel('商機階段', fontsize=11)
ax1.set_title('1. 各階段成交週期分布（堆疊 + 百分比）', fontsize=12, fontweight='bold')
ax1.legend(title='成交週期', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax1.grid(axis='x', alpha=0.3)

# =========== 圖 2：時間趨勢圖 - 訂單數與平均週期（按月） ===========
monthly_data = matched_orders.copy()
monthly_data['order_month'] = monthly_data['order_date'].dt.to_period('M')
monthly_stats = monthly_data.groupby('order_month').agg(
    order_count=('company_id', 'count'),
    avg_cycle=('days_before_order', 'mean')
).reset_index()
monthly_stats['order_month'] = monthly_stats['order_month'].dt.to_timestamp()

ax2_twin = ax2.twinx()
color_bar = '#3498db'
color_line = '#e74c3c'

ax2.bar(monthly_stats['order_month'], monthly_stats['order_count'], 
        color=color_bar, alpha=0.6, label='訂單數', width=20)
ax2_twin.plot(monthly_stats['order_month'], monthly_stats['avg_cycle'], 
              color=color_line, marker='o', linewidth=2, markersize=6, label='平均週期')

ax2.set_xlabel('月份', fontsize=11)
ax2.set_ylabel('訂單數', color=color_bar, fontsize=11)
ax2_twin.set_ylabel('平均週期（天）', color=color_line, fontsize=11)
ax2.tick_params(axis='y', labelcolor=color_bar)
ax2_twin.tick_params(axis='y', labelcolor=color_line)
ax2.set_title('2. 訂單數與平均週期趨勢（按月）', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)

# =========== 圖 3：散佈圖 - 訂單金額 vs 成交週期（按客戶類型著色） ===========
scatter_data = plot_data.dropna(subset=['order_amount'])

# 定義客戶類型的顏色
customer_colors = {
    '新客戶 - 小客戶': '#3498db',  # 藍色
    '新客戶 - 大客戶': '#9b59b6',  # 紫色
    '老客戶 - 小客戶': '#2ecc71',  # 綠色
    '老客戶 - 大客戶': '#e74c3c',  # 紅色
}

for segment, color in customer_colors.items():
    segment_data = scatter_data[scatter_data['customer_segment'] == segment]
    if len(segment_data) > 0:
        ax3.scatter(segment_data['order_amount'], 
                   segment_data['days_before_order'],
                   alpha=0.6, s=30, color=color, 
                   label=segment, edgecolors='white', linewidth=0.5)

# 整體趨勢線
x_val = scatter_data['order_amount'].values
y_val = scatter_data['days_before_order'].values
if len(x_val) > 1:
    z = np.polyfit(x_val, y_val, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(x_val.min(), x_val.max(), 100)
    ax3.plot(x_trend, p(x_trend), "k--", linewidth=2, alpha=0.5, label='整體趨勢')
    
    y_pred = p(x_val)
    ss_res = np.sum((y_val - y_pred) ** 2)
    ss_tot = np.sum((y_val - y_val.mean()) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
    
    ax3.text(0.05, 0.95, f'R² = {r2:.4f}', 
             transform=ax3.transAxes, fontsize=10, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax3.set_xlabel('訂單金額（元）', fontsize=11)
ax3.set_ylabel('成交週期（天）', fontsize=11)
ax3.set_title('3. 訂單金額 vs 成交週期（按客戶類型）', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3)
ax3.legend(fontsize=8, loc='upper right')

# =========== 圖 4：箱型圖 - 各階段 × 客戶類型週期分布 ===========
# 只比較新客戶 vs 老客戶
stage_order = ['A', 'B', 'C1', 'C2', 'D', 'E']
positions = []
labels = []
data_to_plot = []

for i, stage in enumerate(stage_order):
    if stage in plot_data['stage'].unique():
        # 新客戶
        new_data = plot_data[(plot_data['stage'] == stage) & 
                            (plot_data['customer_type'] == '新客戶')]['days_before_order'].values
        if len(new_data) > 0:
            data_to_plot.append(new_data)
            positions.append(i*3)
            labels.append(f'{stage}\n新')
        
        # 老客戶
        old_data = plot_data[(plot_data['stage'] == stage) & 
                            (plot_data['customer_type'] == '老客戶')]['days_before_order'].values
        if len(old_data) > 0:
            data_to_plot.append(old_data)
            positions.append(i*3 + 1)
            labels.append(f'{stage}\n老')

bp = ax4.boxplot(data_to_plot, positions=positions, labels=labels, 
                 patch_artist=True, widths=0.6)

# 新客戶用藍色，老客戶用綠色
for i, patch in enumerate(bp['boxes']):
    if '新' in labels[i]:
        patch.set_facecolor('#3498db')
    else:
        patch.set_facecolor('#2ecc71')
    patch.set_alpha(0.6)

ax4.set_xlabel('商機階段（新/老客戶）', fontsize=11)
ax4.set_ylabel('成交週期（天）', fontsize=11)
ax4.set_title('4. 各階段成交週期分布（新 vs 老客戶）', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# =========== 圖 5：CDF 曲線 - 累積分布 ===========
colors = plt.cm.Set2(np.linspace(0, 1, len(stage_order)))
for idx, stage in enumerate(stage_order):
    if stage in plot_data['stage'].unique():
        stage_data = plot_data[plot_data['stage'] == stage]['days_before_order'].values
        stage_data_sorted = np.sort(stage_data)
        cdf = np.arange(1, len(stage_data_sorted) + 1) / len(stage_data_sorted) * 100
        ax5.plot(stage_data_sorted, cdf, label=f'{stage} ({len(stage_data)}筆)', 
                 linewidth=2, alpha=0.8, color=colors[idx])

ax5.set_xlabel('成交週期（天）', fontsize=11)
ax5.set_ylabel('累積百分比 (%)', fontsize=11)
ax5.set_title('5. 各階段成交週期累積分布（CDF）', fontsize=12, fontweight='bold')
ax5.legend(fontsize=9)
ax5.grid(alpha=0.3)
ax5.set_xlim(0, 365)

# =========== 圖 6：雙軸圖 - 各階段訂單數與平均週期 ===========
x = np.arange(len(stage_stats))
color1 = '#3498db'
bars = ax6.bar(x, stage_stats['order_count'], color=color1, alpha=0.7, label='訂單數', width=0.6)
ax6.set_xlabel('商機階段', fontsize=11)
ax6.set_ylabel('訂單數', color=color1, fontsize=11)
ax6.tick_params(axis='y', labelcolor=color1)
ax6.set_xticks(x)
ax6.set_xticklabels(stage_stats['stage'])

for i, v in enumerate(stage_stats['order_count']):
    ax6.text(i, v + max(stage_stats['order_count'])*0.02, f'{int(v):,}', 
             ha='center', fontsize=9, color=color1)

ax6_twin = ax6.twinx()
color2 = '#e74c3c'
line = ax6_twin.plot(x, stage_stats['avg_days_to_close'],
                     marker='o', color=color2, linewidth=2.5, markersize=8, label='平均週期')
ax6_twin.set_ylabel('平均成交週期（天）', color=color2, fontsize=11)
ax6_twin.tick_params(axis='y', labelcolor=color2)

for i, v in enumerate(stage_stats['avg_days_to_close']):
    ax6_twin.text(i, v + max(stage_stats['avg_days_to_close'])*0.03, f'{int(v)}天', 
                  ha='center', fontsize=9, color=color2)

ax6.set_title('6. 各階段訂單數與平均成交週期', fontsize=12, fontweight='bold')
ax6.grid(axis='y', alpha=0.3)

# 總標題與儲存
plt.suptitle('Phase 4：訂單整合分析 - 完整視覺化（含客戶分類）', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(OUT / "Phase4_Complete_Visualization_v3.png", dpi=200, bbox_inches="tight")
print(f"已儲存完整圖表（6張 + 客戶分類）：Phase4_Complete_Visualization_v3.png")
plt.show()

## 📊 圖 7：商機路徑流轉圖（Sankey Diagram）

展示客戶在各階段之間的流轉路徑與數量

In [ ]:
print("="*70)
print("商機路徑分析 - 完整路徑追蹤")
print("="*70)

# 取得所有成交客戶的 company_id
closed_companies = matched_orders['company_id'].unique()

# 篩選這些客戶的完整商機歷程
stage_history = stage_df[stage_df['company_id'].isin(closed_companies)].copy()
stage_history = stage_history.sort_values(['company_id', 'log_date'])

print(f"\n成交客戶數：{len(closed_companies):,} 家")
print(f"商機歷程記錄：{len(stage_history):,} 筆")

# 計算階段轉換（前一階段 → 當前階段）
stage_history['prev_stage'] = stage_history.groupby('company_id')['stage'].shift(1)

# 移除第一筆（沒有前一階段）
stage_transitions_all = stage_history[stage_history['prev_stage'].notna()].copy()

# 🔥 關鍵改進：排除同階段重複（C1→C1, B→B 等）
stage_transitions = stage_transitions_all[
    stage_transitions_all['prev_stage'] != stage_transitions_all['stage']
].copy()

print(f"\n階段轉換記錄（含重複）：{len(stage_transitions_all):,} 筆")
print(f"階段轉換記錄（排除重複）：{len(stage_transitions):,} 筆")
print(f"同階段重複：{len(stage_transitions_all) - len(stage_transitions):,} 筆 "
      f"({(len(stage_transitions_all) - len(stage_transitions))/len(stage_transitions_all)*100:.1f}%)")

# 建立轉換路徑
stage_transitions['transition'] = (
    stage_transitions['prev_stage'] + ' → ' + stage_transitions['stage']
)

# 統計各種轉換路徑的頻率
transition_counts = stage_transitions['transition'].value_counts()

print("\n前 20 大轉換路徑（排除同階段重複）：")
display(transition_counts.head(20))

# 建立轉換矩陣（用於 Sankey 圖）
transition_matrix = stage_transitions.groupby(['prev_stage', 'stage']).size().reset_index(name='count')
transition_matrix = transition_matrix.sort_values('count', ascending=False)

print("\n轉換矩陣（前 20）：")
display(transition_matrix.head(20))

# ========== 新增：建立每個法人的完整路徑 ==========
print("\n" + "="*70)
print("建立法人完整路徑（排除同階段重複）")
print("="*70)

# 為每個 company_id 建立完整路徑字串
def build_company_path(df):
    """建立單一公司的階段路徑"""
    stages = df.sort_values('log_date')['stage'].tolist()
    
    # 排除連續重複的階段
    path = [stages[0]]
    for stage in stages[1:]:
        if stage != path[-1]:
            path.append(stage)
    
    return ' → '.join(path)

# 建立路徑
company_paths = stage_history.groupby('company_id').apply(build_company_path).reset_index()
company_paths.columns = ['company_id', 'stage_path']

# 統計路徑頻率
path_counts = company_paths['stage_path'].value_counts()

print(f"\n不重複路徑數：{len(path_counts):,} 種")
print(f"\n🔥 前 30 大熱門路徑：")
display(path_counts.head(30))

# 儲存完整路徑與轉換矩陣
company_paths.to_csv(
    OUT / "company_stage_paths.csv",
    index=False,
    encoding="utf-8-sig"
)

path_counts_df = path_counts.reset_index()
path_counts_df.columns = ['stage_path', 'count']
path_counts_df['percentage'] = (path_counts_df['count'] / len(company_paths) * 100).round(2)
path_counts_df.to_csv(
    OUT / "hot_stage_paths.csv",
    index=False,
    encoding="utf-8-sig"
)

transition_matrix.to_csv(
    OUT / "stage_transitions.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n已儲存：")
print(f"  - company_stage_paths.csv（每家公司的完整路徑）")
print(f"  - hot_stage_paths.csv（熱門路徑統計）")
print(f"  - stage_transitions.csv（階段轉換矩陣）")
print("="*70)

---

## 🔍 反向流程分析（階段降級檢測）

檢測 A→C1、B→C2 等反向流程，判斷是否為老客戶重購或記錄錯誤

In [ ]:
print("="*70)
print("反向流程分析（階段降級檢測）")
print("="*70)

# 定義階段順序（數字越小=階段越高）
STAGE_RANK = {"A": 1, "B": 2, "C1": 3, "C2": 4, "D": 5, "E": 6}

# 為 stage_transitions 加上階段排序
stage_transitions['prev_rank'] = stage_transitions['prev_stage'].map(STAGE_RANK)
stage_transitions['curr_rank'] = stage_transitions['stage'].map(STAGE_RANK)

# 計算時間差（兩次記錄的間隔天數）
stage_transitions['time_gap_days'] = (
    stage_transitions['log_date'] - 
    stage_transitions.groupby('company_id')['log_date'].shift(1)
).dt.days

# 判斷流程方向
def classify_transition(row):
    if row['prev_rank'] > row['curr_rank']:
        return '正向晉升'  # E→D, D→C2, C2→C1 等
    elif row['prev_rank'] < row['curr_rank']:
        return '反向降級'  # A→C1, B→C2 等
    else:
        return '同階段'  # 理論上已被過濾

stage_transitions['transition_type'] = stage_transitions.apply(classify_transition, axis=1)

# 統計流程方向
print(f"\n流程方向統計：")
print(stage_transitions['transition_type'].value_counts())
print(f"\n正向晉升率：{(stage_transitions['transition_type'] == '正向晉升').sum() / len(stage_transitions) * 100:.1f}%")
print(f"反向降級率：{(stage_transitions['transition_type'] == '反向降級').sum() / len(stage_transitions) * 100:.1f}%")

# ========== 重點：反向流程分析 ==========
reverse_transitions = stage_transitions[stage_transitions['transition_type'] == '反向降級'].copy()

print(f"\n" + "="*70)
print(f"反向流程詳細分析（共 {len(reverse_transitions):,} 筆）")
print("="*70)

# 反向路徑統計
print(f"\n前 20 大反向路徑：")
reverse_paths = reverse_transitions['transition'].value_counts().head(20)
display(reverse_paths)

# 與訂單資料比對：檢查是否為老客戶
reverse_companies = reverse_transitions['company_id'].unique()
print(f"\n涉及反向流程的公司：{len(reverse_companies):,} 家")

# 檢查這些公司的訂單數（判斷是否為重購）
reverse_order_stats = matched_orders[matched_orders['company_id'].isin(reverse_companies)].groupby('company_id').agg(
    order_count=('order_date', 'count'),
    first_order=('order_date', 'min'),
    last_order=('order_date', 'max')
).reset_index()

multi_order_companies = reverse_order_stats[reverse_order_stats['order_count'] > 1]
print(f"其中多筆訂單客戶：{len(multi_order_companies):,} 家 ({len(multi_order_companies)/len(reverse_companies)*100:.1f}%)")
print(f"單筆訂單客戶：{len(reverse_order_stats) - len(multi_order_companies):,} 家")

# 時間間隔分析
print(f"\n反向流程的時間間隔分析：")
time_gap_stats = reverse_transitions['time_gap_days'].describe()
display(time_gap_stats)

print(f"\n時間間隔分組：")
reverse_transitions['time_gap_group'] = pd.cut(
    reverse_transitions['time_gap_days'],
    bins=[0, 7, 30, 90, 180, 9999],
    labels=['<7天', '7-30天', '30-90天', '90-180天', '>180天']
)
time_gap_dist = reverse_transitions['time_gap_group'].value_counts().sort_index()
display(time_gap_dist)

# 判斷：短時間間隔（<7天）可能是記錄錯誤，長時間間隔（>90天）可能是重購
short_interval = reverse_transitions[reverse_transitions['time_gap_days'] < 7]
long_interval = reverse_transitions[reverse_transitions['time_gap_days'] > 90]

print(f"\n" + "="*70)
print("判斷結果：")
print("="*70)

print(f"\n1. 可能的記錄錯誤（<7天）：")
print(f"   數量：{len(short_interval):,} 筆 ({len(short_interval)/len(reverse_transitions)*100:.1f}%)")
print(f"   主要路徑：{short_interval['transition'].value_counts().head(5).to_dict()}")

print(f"\n2. 可能的老客戶重購（>90天）：")
print(f"   數量：{len(long_interval):,} 筆 ({len(long_interval)/len(reverse_transitions)*100:.1f}%)")
print(f"   主要路徑：{long_interval['transition'].value_counts().head(5).to_dict()}")

print(f"\n3. 中等間隔（7-90天）：")
mid_interval = reverse_transitions[
    (reverse_transitions['time_gap_days'] >= 7) & 
    (reverse_transitions['time_gap_days'] <= 90)
]
print(f"   數量：{len(mid_interval):,} 筆 ({len(mid_interval)/len(reverse_transitions)*100:.1f}%)")
print(f"   可能原因：新專案評估、階段調整、業務追蹤")

# ========== 單階段成交分析 ==========
print(f"\n" + "="*70)
print("單階段成交分析")
print("="*70)

single_stage_paths = company_paths[~company_paths['stage_path'].str.contains('→')].copy()
multi_stage_paths = company_paths[company_paths['stage_path'].str.contains('→')].copy()

print(f"\n單階段成交：{len(single_stage_paths):,} 家 ({len(single_stage_paths)/len(company_paths)*100:.1f}%)")
print(f"多階段流轉：{len(multi_stage_paths):,} 家 ({len(multi_stage_paths)/len(company_paths)*100:.1f}%)")

print(f"\n單階段成交分布：")
single_stage_dist = single_stage_paths['stage_path'].value_counts()
display(single_stage_dist)

# 交叉比對：單階段 vs 新老客戶
single_stage_companies = single_stage_paths['company_id'].values
single_stage_orders = matched_orders[matched_orders['company_id'].isin(single_stage_companies)].copy()

print(f"\n單階段成交的客戶類型：")
print(single_stage_orders['customer_type'].value_counts())
print(f"\n單階段成交的客戶規模：")
print(single_stage_orders['customer_size'].value_counts())

# 對比：多階段成交
multi_stage_companies = multi_stage_paths['company_id'].values
multi_stage_orders = matched_orders[matched_orders['company_id'].isin(multi_stage_companies)].copy()

print(f"\n多階段成交的客戶類型：")
print(multi_stage_orders['customer_type'].value_counts())
print(f"\n多階段成交的客戶規模：")
print(multi_stage_orders['customer_size'].value_counts())

# ========== 儲存分析結果 ==========
reverse_transitions.to_csv(
    OUT / "reverse_transitions.csv",
    index=False,
    encoding="utf-8-sig"
)

single_vs_multi = pd.DataFrame({
    'path_type': ['單階段', '多階段'],
    'company_count': [len(single_stage_paths), len(multi_stage_paths)],
    'percentage': [
        len(single_stage_paths)/len(company_paths)*100,
        len(multi_stage_paths)/len(company_paths)*100
    ]
})
single_vs_multi.to_csv(
    OUT / "single_vs_multi_stage.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n已儲存：")
print(f"  - reverse_transitions.csv（反向流程明細）")
print(f"  - single_vs_multi_stage.csv（單階段 vs 多階段統計）")

print("\n" + "="*70)
print("反向流程分析完成！")
print("="*70)

---

## 🎯 成交路徑分析（Order Conversion Paths）

追蹤每筆訂單從第一次接觸到最終成交的完整商機歷程

In [ ]:
print("="*70)
print("成交路徑分析 - 每筆訂單的完整成交歷程")
print("="*70)

# 為每筆訂單建立成交路徑
# 思路：matched_orders 每一列是一筆訂單，找出該訂單成交前的完整商機歷程

order_conversion_paths = []

for idx, order in matched_orders.iterrows():
    company_id = order['company_id']
    order_date = order['order_date']
    final_stage = order['stage']  # 成交階段
    order_amount = order['order_amount']
    
    # 找出該公司在成交日期之前的所有商機記錄
    company_history = stage_df[
        (stage_df['company_id'] == company_id) & 
        (stage_df['log_date'] <= order_date)
    ].sort_values('log_date')
    
    if len(company_history) > 0:
        # 建立路徑（排除連續重複階段）
        stages = company_history['stage'].tolist()
        path = [stages[0]]
        for stage in stages[1:]:
            if stage != path[-1]:
                path.append(stage)
        
        # 加上成交標記
        path_str = ' → '.join(path) + f' 【{final_stage}成交】'
        
        order_conversion_paths.append({
            'company_id': company_id,
            'order_date': order_date,
            'order_amount': order_amount,
            'final_stage': final_stage,
            'conversion_path': path_str,
            'path_length': len(path),
            'first_stage': path[0],
            'customer_type': order['customer_type'],
            'customer_size': order['customer_size']
        })

# 轉換為 DataFrame
order_paths_df = pd.DataFrame(order_conversion_paths)

print(f"\n成交訂單數：{len(order_paths_df):,} 筆")
print(f"\n前 20 大熱門成交路徑：")
conversion_path_counts = order_paths_df['conversion_path'].value_counts()
display(conversion_path_counts.head(20))

# 統計路徑長度分布
print(f"\n" + "="*70)
print("成交路徑長度分布")
print("="*70)

path_length_dist = order_paths_df['path_length'].value_counts().sort_index()
print(f"\n路徑長度統計：")
for length, count in path_length_dist.items():
    percentage = count / len(order_paths_df) * 100
    print(f"  {length} 個階段：{count:,} 筆訂單 ({percentage:.1f}%)")

# 按成交階段分組
print(f"\n" + "="*70)
print("各成交階段的平均路徑長度")
print("="*70)

stage_path_stats = order_paths_df.groupby('final_stage').agg(
    order_count=('company_id', 'count'),
    avg_path_length=('path_length', 'mean'),
    median_path_length=('path_length', 'median'),
    min_path_length=('path_length', 'min'),
    max_path_length=('path_length', 'max')
).reset_index()

# 排序
stage_path_stats['stage_rank'] = stage_path_stats['final_stage'].map(STAGE_ORDER)
stage_path_stats = stage_path_stats.sort_values('stage_rank')

print(f"\n成交階段 vs 路徑長度：")
display(stage_path_stats[['final_stage', 'order_count', 'avg_path_length', 'median_path_length', 'min_path_length', 'max_path_length']])

# 分析最短路徑（直接成交）vs 最長路徑（完整培養）
print(f"\n" + "="*70)
print("極端案例分析")
print("="*70)

# 路徑長度 = 1（單階段直接成交）
direct_conversion = order_paths_df[order_paths_df['path_length'] == 1]
print(f"\n單階段直接成交（路徑長度=1）：")
print(f"  訂單數：{len(direct_conversion):,} 筆 ({len(direct_conversion)/len(order_paths_df)*100:.1f}%)")
print(f"  總金額：{direct_conversion['order_amount'].sum():,.0f} 元")
print(f"  平均金額：{direct_conversion['order_amount'].mean():,.0f} 元")
print(f"  新客戶：{(direct_conversion['customer_type'] == '新客戶').sum():,} 筆")
print(f"  老客戶：{(direct_conversion['customer_type'] == '老客戶').sum():,} 筆")

# 路徑長度 >= 5（完整培養）
long_conversion = order_paths_df[order_paths_df['path_length'] >= 5]
print(f"\n完整培養路徑（路徑長度≥5）：")
print(f"  訂單數：{len(long_conversion):,} 筆 ({len(long_conversion)/len(order_paths_df)*100:.1f}%)")
print(f"  總金額：{long_conversion['order_amount'].sum():,.0f} 元")
print(f"  平均金額：{long_conversion['order_amount'].mean():,.0f} 元")
print(f"  新客戶：{(long_conversion['customer_type'] == '新客戶').sum():,} 筆")
print(f"  老客戶：{(long_conversion['customer_type'] == '老客戶').sum():,} 筆")

# 最常見的起始階段 → 成交階段組合
print(f"\n" + "="*70)
print("起始階段 → 成交階段 組合（前 15 名）")
print("="*70)

start_end_combo = order_paths_df.groupby(['first_stage', 'final_stage']).size().reset_index(name='count')
start_end_combo = start_end_combo.sort_values('count', ascending=False)

print(f"\n最常見的起點→終點：")
for idx, row in start_end_combo.head(15).iterrows():
    percentage = row['count'] / len(order_paths_df) * 100
    print(f"  {row['first_stage']:3s} → {row['final_stage']:3s}  {row['count']:6,} 筆 ({percentage:5.1f}%)")

# 儲存成交路徑資料
order_paths_df.to_csv(
    OUT / "order_conversion_paths.csv",
    index=False,
    encoding="utf-8-sig"
)

# 儲存成交路徑統計
conversion_path_stats = conversion_path_counts.reset_index()
conversion_path_stats.columns = ['conversion_path', 'count']
conversion_path_stats['percentage'] = (conversion_path_stats['count'] / len(order_paths_df) * 100).round(2)
conversion_path_stats.to_csv(
    OUT / "hot_conversion_paths.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n已儲存：")
print(f"  - order_conversion_paths.csv（每筆訂單的完整成交路徑）")
print(f"  - hot_conversion_paths.csv（熱門成交路徑統計）")

print("\n" + "="*70)
print("成交路徑分析完成！")
print("="*70)

---

## 🌳 樹狀路徑分析（Tree Path Analysis）

建立樹狀結構展示所有可能的成交路徑，並標記：
- ✅ **成交路徑**：最終成交的路徑
- 🔄 **再購路徑**：老客戶重複購買的路徑

In [ ]:
print("="*70)
print("樹狀路徑分析 - 成交路徑 & 再購路徑")
print("="*70)

# ========== 步驟 1：建立樹狀路徑結構 ==========
from collections import defaultdict

# 為每筆訂單標記是否為再購
order_paths_df['is_repurchase'] = order_paths_df['customer_type'] == '老客戶'

# 建立樹狀結構（每個節點記錄到達該節點的訂單）
tree_paths = defaultdict(lambda: {
    'total_orders': 0,
    'repurchase_orders': 0,
    'new_customer_orders': 0,
    'total_amount': 0,
    'final_stages': defaultdict(int),
    'paths': []
})

for idx, row in order_paths_df.iterrows():
    path_str = row['conversion_path']
    # 移除成交標記，只保留路徑
    path_clean = path_str.split('【')[0].strip()
    stages = [s.strip() for s in path_clean.split('→')]
    
    # 記錄完整路徑
    full_path = ' → '.join(stages)
    tree_paths[full_path]['total_orders'] += 1
    tree_paths[full_path]['total_amount'] += row['order_amount']
    tree_paths[full_path]['final_stages'][row['final_stage']] += 1
    
    if row['is_repurchase']:
        tree_paths[full_path]['repurchase_orders'] += 1
    else:
        tree_paths[full_path]['new_customer_orders'] += 1
    
    # 也記錄部分路徑（用於樹狀圖）
    for i in range(1, len(stages) + 1):
        partial_path = ' → '.join(stages[:i])
        if partial_path != full_path:  # 避免重複計算完整路徑
            tree_paths[partial_path]['total_orders'] += 1
            tree_paths[partial_path]['total_amount'] += row['order_amount']
            
            if row['is_repurchase']:
                tree_paths[partial_path]['repurchase_orders'] += 1
            else:
                tree_paths[partial_path]['new_customer_orders'] += 1

# ========== 步驟 2：統計樹狀路徑 ==========
print(f"\n總路徑數（含部分路徑）：{len(tree_paths):,}")

# 轉換為 DataFrame
tree_data = []
for path, stats in tree_paths.items():
    stages = path.split(' → ')
    path_length = len(stages)
    
    # 判斷是否為成交路徑（完整路徑，不是部分路徑）
    is_leaf = len(stats['final_stages']) > 0
    
    tree_data.append({
        'path': path,
        'path_length': path_length,
        'last_stage': stages[-1],
        'is_leaf': is_leaf,
        'total_orders': stats['total_orders'],
        'new_customer_orders': stats['new_customer_orders'],
        'repurchase_orders': stats['repurchase_orders'],
        'repurchase_rate': stats['repurchase_orders'] / stats['total_orders'] * 100 if stats['total_orders'] > 0 else 0,
        'total_amount': stats['total_amount'],
        'avg_amount': stats['total_amount'] / stats['total_orders'] if stats['total_orders'] > 0 else 0,
        'final_stages': dict(stats['final_stages'])
    })

tree_df = pd.DataFrame(tree_data)
tree_df = tree_df.sort_values(['path_length', 'total_orders'], ascending=[True, False])

# ========== 步驟 3：分析成交路徑（Leaf Nodes）==========
print(f"\n" + "="*70)
print("成交路徑分析（完整路徑，最終成交）")
print("="*70)

leaf_paths = tree_df[tree_df['is_leaf'] == True].copy()
print(f"\n成交路徑總數：{len(leaf_paths):,} 種")
print(f"涵蓋訂單：{leaf_paths['total_orders'].sum():,} 筆")

print(f"\n前 30 大成交路徑：")
top_leaf_paths = leaf_paths.nlargest(30, 'total_orders')
for idx, row in top_leaf_paths.iterrows():
    repurchase_marker = '🔄' if row['repurchase_rate'] > 50 else ''
    new_customer_marker = '✨' if row['repurchase_rate'] < 50 else ''
    marker = repurchase_marker or new_customer_marker
    
    print(f"{marker} {row['path']:50s}  "
          f"{int(row['total_orders']):4d} 筆  "
          f"(新:{int(row['new_customer_orders']):3d} / 購:{int(row['repurchase_orders']):3d})  "
          f"再購率:{row['repurchase_rate']:5.1f}%")

# ========== 步驟 4：分析再購路徑 ==========
print(f"\n" + "="*70)
print("再購路徑分析（老客戶重複購買）")
print("="*70)

repurchase_dominated = leaf_paths[leaf_paths['repurchase_rate'] >= 70].copy()
print(f"\n高再購路徑（再購率≥70%）：{len(repurchase_dominated):,} 種")
print(f"涵蓋訂單：{int(repurchase_dominated['total_orders'].sum()):,} 筆")

if len(repurchase_dominated) > 0:
    print(f"\n前 20 大高再購路徑：")
    for idx, row in repurchase_dominated.nlargest(20, 'total_orders').iterrows():
        print(f"🔄 {row['path']:50s}  "
              f"{int(row['total_orders']):4d} 筆  "
              f"再購率:{row['repurchase_rate']:5.1f}%  "
              f"平均金額:{row['avg_amount']:,.0f}")

# ========== 步驟 5：分析新客戶路徑 ==========
print(f"\n" + "="*70)
print("新客戶路徑分析")
print("="*70)

new_customer_dominated = leaf_paths[leaf_paths['repurchase_rate'] <= 30].copy()
print(f"\n高新客路徑（新客率≥70%）：{len(new_customer_dominated):,} 種")
print(f"涵蓋訂單：{int(new_customer_dominated['total_orders'].sum()):,} 筆")

if len(new_customer_dominated) > 0:
    print(f"\n前 20 大高新客路徑：")
    for idx, row in new_customer_dominated.nlargest(20, 'total_orders').iterrows():
        print(f"✨ {row['path']:50s}  "
              f"{int(row['total_orders']):4d} 筆  "
              f"新客率:{100-row['repurchase_rate']:5.1f}%  "
              f"平均金額:{row['avg_amount']:,.0f}")

# ========== 步驟 6：路徑長度分析 ==========
print(f"\n" + "="*70)
print("路徑長度分析")
print("="*70)

length_analysis = leaf_paths.groupby('path_length').agg(
    path_count=('path', 'count'),
    total_orders=('total_orders', 'sum'),
    avg_repurchase_rate=('repurchase_rate', 'mean'),
    avg_amount=('avg_amount', 'mean')
).reset_index()

print(f"\n各長度路徑統計：")
for idx, row in length_analysis.iterrows():
    print(f"  {int(row['path_length'])} 階段：{int(row['path_count']):4d} 種路徑  "
          f"{int(row['total_orders']):5d} 筆訂單  "
          f"平均再購率:{row['avg_repurchase_rate']:5.1f}%  "
          f"平均金額:{row['avg_amount']:,.0f}")

# ========== 步驟 7：儲存分析結果 ==========
tree_df.to_csv(
    OUT / "tree_path_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

leaf_paths.to_csv(
    OUT / "conversion_leaf_paths.csv",
    index=False,
    encoding="utf-8-sig"
)

repurchase_dominated.to_csv(
    OUT / "repurchase_dominated_paths.csv",
    index=False,
    encoding="utf-8-sig"
)

new_customer_dominated.to_csv(
    OUT / "new_customer_dominated_paths.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\n已儲存：")
print(f"  - tree_path_analysis.csv（完整樹狀路徑）")
print(f"  - conversion_leaf_paths.csv（所有成交路徑）")
print(f"  - repurchase_dominated_paths.csv（高再購路徑）")
print(f"  - new_customer_dominated_paths.csv（高新客路徑）")

print("\n" + "="*70)
print("樹狀路徑分析完成！")
print("="*70)

In [ ]:
try:
    import plotly.graph_objects as go
    import plotly.express as px
    
    print("="*70)
    print("繪製樹狀路徑 Sunburst 圖（互動式）")
    print("="*70)
    
    # ========== 準備 Sunburst 數據（簡化版）==========
    # 只使用完整成交路徑（葉子節點）來建立樹狀結構
    
    sunburst_data = []
    
    # 添加根節點
    sunburst_data.append({
        'ids': '',
        'labels': '全部成交路徑',
        'parents': '',
        'values': int(leaf_paths['total_orders'].sum()),
        'text': f"總訂單：{int(leaf_paths['total_orders'].sum()):,}",
        'color': 'lightgray'
    })
    
    # 建立階層結構：根節點 -> 第一階段 -> ... -> 最後階段
    added_nodes = set([''])  # 已添加的節點ID
    
    for idx, row in leaf_paths.nlargest(100, 'total_orders').iterrows():  # 只取前100條路徑
        path = row['path']
        stages = path.split(' → ')
        
        # 為每個階段建立節點
        for i in range(len(stages)):
            # 當前節點的完整路徑作為唯一ID
            current_id = ' → '.join(stages[:i+1])
            parent_id = ' → '.join(stages[:i]) if i > 0 else ''
            
            if current_id not in added_nodes:
                # 判斷節點顏色
                if i == len(stages) - 1:  # 最後一個節點（葉子）
                    # 根據再購率著色
                    if row['repurchase_rate'] >= 70:
                        color = 'rgba(231, 76, 60, 0.8)'  # 紅色 - 高再購
                        marker = '🔄'
                    elif row['repurchase_rate'] <= 30:
                        color = 'rgba(52, 152, 219, 0.8)'  # 藍色 - 高新客
                        marker = '✨'
                    else:
                        color = 'rgba(241, 196, 15, 0.8)'  # 黃色 - 混合
                        marker = '🟡'
                    
                    text = (f"{marker} {stages[i]}<br>"
                           f"訂單：{int(row['total_orders']):,}<br>"
                           f"新客：{int(row['new_customer_orders']):,}<br>"
                           f"再購：{int(row['repurchase_orders']):,}<br>"
                           f"再購率：{row['repurchase_rate']:.1f}%")
                    value = int(row['total_orders'])
                else:
                    # 中間節點
                    color = 'rgba(200, 200, 200, 0.4)'
                    text = f"{stages[i]}"
                    value = int(row['total_orders'])
                
                sunburst_data.append({
                    'ids': current_id,
                    'labels': stages[i],
                    'parents': parent_id,
                    'values': value,
                    'text': text,
                    'color': color
                })
                
                added_nodes.add(current_id)
    
    # 轉換為 DataFrame
    sunburst_df = pd.DataFrame(sunburst_data)
    
    print(f"\n節點總數：{len(sunburst_df):,}")
    print(f"成交路徑：{len(leaf_paths.nlargest(100, 'total_orders')):,} 條（前100大）")
    
    # ========== 建立 Sunburst 圖 ==========
    fig = go.Figure(go.Sunburst(
        ids=sunburst_df['ids'],
        labels=sunburst_df['labels'],
        parents=sunburst_df['parents'],
        values=sunburst_df['values'],
        text=sunburst_df['text'],
        hovertext=sunburst_df['text'],
        hoverinfo='text',
        marker=dict(
            colors=sunburst_df['color'],
            line=dict(color='white', width=2)
        ),
        branchvalues='total'
    ))
    
    fig.update_layout(
        title=dict(
            text="成交路徑樹狀圖（Sunburst）- 前100大路徑<br>"
                 "<sub>🔴 紅色=高再購路徑(≥70%) | 🔵 藍色=高新客路徑(≤30%) | 🟡 黃色=混合路徑 | ⚪ 灰色=中間節點</sub>",
            font=dict(size=16, family="Microsoft JhengHei"),
            x=0.5,
            xanchor='center'
        ),
        font=dict(size=12, family="Microsoft JhengHei"),
        height=800,
        width=900,
        margin=dict(l=20, r=20, t=120, b=20)
    )
    
    # 儲存為 HTML
    output_path = OUT / "Tree_Path_Sunburst.html"
    fig.write_html(output_path)
    print(f"\n✅ 已儲存：Tree_Path_Sunburst.html")
    print(f"   路徑：{output_path}")
    print(f"   可在瀏覽器中開啟查看互動式樹狀圖")
    
    # 統計資訊
    print(f"\n📊 Sunburst 圖統計：")
    print(f"   總節點數：{len(sunburst_df):,}")
    print(f"   涵蓋訂單數：{int(leaf_paths.nlargest(100, 'total_orders')['total_orders'].sum()):,}")
    print(f"   佔總訂單比例：{leaf_paths.nlargest(100, 'total_orders')['total_orders'].sum() / leaf_paths['total_orders'].sum() * 100:.1f}%")
    
    # 顏色分布
    top_100 = leaf_paths.nlargest(100, 'total_orders')
    high_repurchase = len(top_100[top_100['repurchase_rate'] >= 70])
    high_new = len(top_100[top_100['repurchase_rate'] <= 30])
    mixed = len(top_100) - high_repurchase - high_new
    
    print(f"\n🎨 路徑類型分布（前100大）：")
    print(f"   🔴 高再購路徑：{high_repurchase:,} 種 ({high_repurchase/len(top_100)*100:.1f}%)")
    print(f"   🔵 高新客路徑：{high_new:,} 種 ({high_new/len(top_100)*100:.1f}%)")
    print(f"   🟡 混合路徑：{mixed:,} 種 ({mixed/len(top_100)*100:.1f}%)")
    
    print("\n" + "="*70)
    
    # 顯示圖表
    fig.show()
    
except ImportError:
    print("="*70)
    print("⚠️ 未安裝 plotly，跳過 Sunburst 圖")
    print("="*70)
    print("\n安裝方式：pip install plotly")
    print("\n可在安裝後重新執行此 cell 來生成互動式樹狀圖")
    print("="*70)
except Exception as e:
    print("="*70)
    print(f"⚠️ 錯誤：{e}")
    print("="*70)
    import traceback
    traceback.print_exc()

In [ ]:
try:
    import plotly.graph_objects as go
    
    print("="*70)
    print("繪製商機路徑 Sankey 圖（排除同階段重複）")
    print("="*70)
    
    # transition_matrix 已經排除了同階段重複（在 Cell-24 建立）
    # 只保留轉換次數 >= 10 的路徑（避免圖表過於複雜）
    major_transitions = transition_matrix[transition_matrix['count'] >= 10].copy()
    
    print(f"\n總轉換路徑：{len(transition_matrix)} 種")
    print(f"主要轉換路徑（≥10次）：{len(major_transitions)} 種")
    print(f"涵蓋轉換數：{major_transitions['count'].sum():,} 次")
    print(f"涵蓋率：{major_transitions['count'].sum() / transition_matrix['count'].sum() * 100:.1f}%")
    
    # 定義階段順序（從高到低）
    stage_order_map = {'E': 6, 'D': 5, 'C2': 4, 'C1': 3, 'B': 2, 'A': 1}
    
    # 建立節點列表（按階段順序排列）
    all_stages = sorted(
        list(set(major_transitions['prev_stage'].tolist() + major_transitions['stage'].tolist())),
        key=lambda x: stage_order_map.get(x, 0),
        reverse=True
    )
    stage_to_idx = {stage: idx for idx, stage in enumerate(all_stages)}
    
    print(f"\n節點順序：{' → '.join(all_stages)}")
    
    # 建立 Sankey 數據
    source = [stage_to_idx[s] for s in major_transitions['prev_stage']]
    target = [stage_to_idx[s] for s in major_transitions['stage']]
    value = major_transitions['count'].tolist()
    
    # 定義顏色（階段越高，顏色越深）
    stage_colors = {
        'A': 'rgba(231, 76, 60, 0.9)',   # 紅色（最高階段）
        'B': 'rgba(230, 126, 34, 0.9)',  # 橘色
        'C1': 'rgba(241, 196, 15, 0.9)', # 黃色
        'C2': 'rgba(46, 204, 113, 0.9)', # 綠色
        'D': 'rgba(52, 152, 219, 0.9)',  # 藍色
        'E': 'rgba(155, 89, 182, 0.9)',  # 紫色（最低階段）
    }
    
    node_colors = [stage_colors.get(s, 'rgba(128, 128, 128, 0.8)') for s in all_stages]
    
    # 為連結著色（根據來源階段）
    link_colors = [stage_colors.get(major_transitions.iloc[i]['prev_stage'], 'rgba(128, 128, 128, 0.3)').replace('0.9', '0.3') 
                   for i in range(len(major_transitions))]
    
    # 建立節點標籤（顯示階段名稱 + 流入/流出總數）
    node_labels = []
    for stage in all_stages:
        inflow = major_transitions[major_transitions['stage'] == stage]['count'].sum()
        outflow = major_transitions[major_transitions['prev_stage'] == stage]['count'].sum()
        node_labels.append(f"{stage}<br>進:{inflow:,} / 出:{outflow:,}")
    
    # 建立 Sankey 圖
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=20,
            thickness=25,
            line=dict(color="white", width=2),
            label=node_labels,
            color=node_colors,
            customdata=all_stages,
            hovertemplate='階段 %{customdata}<br>總流入: %{value}<extra></extra>'
        ),
        link=dict(
            source=source,
            target=target,
            value=value,
            color=link_colors,
            customdata=[f"{major_transitions.iloc[i]['prev_stage']} → {major_transitions.iloc[i]['stage']}" 
                       for i in range(len(major_transitions))],
            hovertemplate='路徑: %{customdata}<br>次數: %{value:,}<extra></extra>'
        )
    )])
    
    fig.update_layout(
        title=dict(
            text="商機階段流轉路徑圖（Sankey Diagram）<br><sub>僅顯示轉換次數 ≥ 10 的真實階段變化（已排除同階段重複）</sub>",
            font=dict(size=18, family="Microsoft JhengHei"),
            x=0.5,
            xanchor='center'
        ),
        font=dict(size=13, family="Microsoft JhengHei"),
        height=700,
        margin=dict(l=50, r=50, t=100, b=50),
        paper_bgcolor='rgba(245, 245, 245, 1)',
        plot_bgcolor='rgba(255, 255, 255, 1)'
    )
    
    # 儲存為 HTML（互動式）
    output_path = OUT / "Stage_Flow_Sankey.html"
    fig.write_html(output_path)
    print(f"\n✅ 已儲存：Stage_Flow_Sankey.html")
    print(f"   路徑：{output_path}")
    print(f"   可在瀏覽器中開啟查看互動式圖表")
    
    # 統計資訊
    print(f"\n📊 Sankey 圖統計：")
    print(f"   節點數：{len(all_stages)} 個階段")
    print(f"   連結數：{len(major_transitions)} 種路徑")
    print(f"   總流量：{sum(value):,} 次轉換")
    
    # 顯示最熱門的 5 條路徑
    print(f"\n🔥 最熱門的 5 條路徑：")
    for idx, row in major_transitions.head(5).iterrows():
        print(f"   {row['prev_stage']:3s} → {row['stage']:3s}  {row['count']:6,} 次 "
              f"({row['count']/major_transitions['count'].sum()*100:5.1f}%)")
    
    print("\n" + "="*70)
    
    # 顯示圖表
    fig.show()
    
except ImportError:
    print("="*70)
    print("⚠️ 未安裝 plotly，跳過 Sankey 圖")
    print("="*70)
    print("\n安裝方式：pip install plotly")
    print("\n可在安裝後重新執行此 cell 來生成互動式路徑圖")
    print("="*70)
except NameError as e:
    print("="*70)
    print(f"⚠️ 錯誤：{e}")
    print("="*70)
    print("\n請確保已執行前面的商機路徑分析 cell（Cell-24）")
    print("transition_matrix 變數尚未定義")
    print("="*70)

---

# 總結報告

In [ ]:
print("="*70)
print("Phase 4（時間序列版）執行摘要")
print("="*70)

print(f"\n輸入資料：")
print(f"  訂單總數：{len(orders_df):,} 筆")
print(f"  商機日誌：{len(stage_df):,} 筆")
print(f"  涵蓋公司：{stage_df['company_id'].nunique():,} 家")

print(f"\n時間序列匹配：")
print(f"  成功匹配：{len(matched_orders):,} 筆 ({len(matched_orders)/len(orders_df)*100:.1f}%)")
print(f"  無法匹配：{unmatched:,} 筆 ({unmatched/len(orders_df)*100:.1f}%)")
print(f"  匹配總金額：{matched_orders['order_amount'].sum():,.0f} 元")

print(f"\n成交週期統計：")
print(f"  平均週期：{matched_orders['days_before_order'].mean():.0f} 天")
print(f"  中位數週期：{matched_orders['days_before_order'].median():.0f} 天")
print(f"  最快成交：{matched_orders['days_before_order'].min():.0f} 天 (同日成交)")
print(f"  最慢成交：{matched_orders['days_before_order'].max():.0f} 天")

print(f"\n各階段平均週期（天）：")
for _, row in stage_stats.iterrows():
    print(f"  {row['stage']:4s}: {row['avg_days_to_close']:6.0f} 天 "
          f"(中位數 {row['median_days_to_close']:3.0f} 天) "
          f"- {row['order_count']:,} 筆訂單")

print(f"\n快速成交分析 (<30天)：")
quick_deals = matched_orders[matched_orders['days_before_order'] < 30]
print(f"  快速成交：{len(quick_deals):,} 筆 ({len(quick_deals)/len(matched_orders)*100:.1f}%)")
print(f"  主要階段：{quick_deals['stage'].value_counts().head(3).to_dict()}")

print(f"\n超長週期分析 (>180天)：")
long_deals = matched_orders[matched_orders['days_before_order'] > 180]
print(f"  超長週期：{len(long_deals):,} 筆 ({len(long_deals)/len(matched_orders)*100:.1f}%)")
print(f"  主要階段：{long_deals['stage'].value_counts().head(3).to_dict()}")

print(f"\n輸出目錄：{OUT}")
print(f"\n輸出檔案：")
for f in sorted(OUT.glob("*")):
    print(f"  {f.name}")

print("\n="*70)
print("Phase 4（時間序列版）分析完成！")
print("="*70)